# Artificial Intelligence — Lab 2
## Breadth-First Search and Depth-First Search

**Course Learning Outcome — CLO2**  
Determine appropriate uninformed search strategies and explain their behavior.

**Environment:** Python 3 / Jupyter Notebook  
**Submission:** completed notebook containing predictions, traces, code, experimental results, justifications, debugging answers, and reflection.

> **Assessment principle:** Working code is only one part of the evidence. Most marks come from your ability to **predict, trace, justify, compare, modify, and explain** BFS and DFS.

## Lab at a Glance

| Stage | Suggested time | What you will do |
|---|---:|---|
| 1. Search recap & prediction | 10 min | Reason about frontier behavior before coding |
| 2. BFS implementation & trace | 30 min | Implement queue-based graph search |
| 3. DFS implementation & trace | 25 min | Implement stack-based graph search |
| 4. Grid experiment & comparison | 30 min | Compare paths and explored states |
| 5. Debugging & personalized variation | 15 min | Diagnose common errors and test a variant |
| 6. Reflection & individual explanation | 10 min | Defend conclusions and connect to theory |

> **Main idea:** BFS and DFS may explore the same state space, but their **frontier discipline** produces different search behavior.

## Learning Objectives

By the end of this lab, you should be able to:

1. explain the role of the **frontier** and **explored set**;
2. predict the expansion order of BFS and DFS on a small graph;
3. implement BFS using a **FIFO queue**;
4. implement DFS using a **LIFO stack**;
5. reconstruct a solution path using parent links;
6. compare BFS and DFS in terms of completeness, solution depth, explored states, and memory;
7. explain why graph search needs duplicate-state handling;
8. diagnose common implementation errors;
9. justify which strategy is more appropriate for a given problem.

In [ ]:
from collections import deque
from typing import Dict, List, Tuple, Set, Optional

print("Lab 2 environment ready.")

# Part I — Search Mechanics Before Coding

We will first use the graph below.

```text
          S
       /  |  \
      A   B   C
     / \\  |    \\
    D   E F     G
         \\     /
          \\___/
```

Adjacency order is important:

```python
S: A, B, C
A: D, E
B: F
C: G
E: G
F: G
```

Assume graph search checks repeated states.

In [ ]:
GRAPH = {
    "S": ["A", "B", "C"],
    "A": ["D", "E"],
    "B": ["F"],
    "C": ["G"],
    "D": [],
    "E": ["G"],
    "F": ["G"],
    "G": [],
}
START = "S"
GOAL = "G"

## Task 1.1 — Predict BFS Before Running It

Breadth-First Search expands the **shallowest frontier node first**.

Without executing any search code, complete the trace.

| Step | Expanded node | Frontier after expansion | Explored |
|---:|---|---|---|
| 0 | — | `[S]` | `{}` |
| 1 |  |  |  |
| 2 |  |  |  |
| 3 |  |  |  |
| 4 |  |  |  |

Then answer:

1. Which node do you predict BFS will expand immediately before first discovering `G`?
2. What solution path do you predict BFS will return?
3. Why is a **queue** appropriate for BFS?

**Your answers:**

## Task 1.2 — Predict DFS Before Running It

Depth-First Search expands the **most recently added frontier node first**.

For this lab, DFS should explore neighbors from **left to right** according to the adjacency list. To achieve that with a Python stack, think carefully about the order in which neighbors must be pushed.

Complete the trace:

| Step | Expanded node | Stack after expansion | Explored |
|---:|---|---|---|
| 0 | — | `[S]` | `{}` |
| 1 |  |  |  |
| 2 |  |  |  |
| 3 |  |  |  |
| 4 |  |  |  |

Then answer:

1. Which branch do you predict DFS will pursue first?
2. What path do you predict it may return?
3. Why can DFS return a different solution from BFS even on the same graph?

**Your answers:**

# Part II — Breadth-First Search

A graph-search implementation needs more than a frontier. It must also avoid repeatedly exploring the same state.

For this lab, your BFS function must return:

```python
(path, expansion_order)
```

where:

- `path` is a list of states from start to goal;
- `expansion_order` records the order in which states were removed from the frontier for expansion.

In [ ]:
def reconstruct_path(parent: Dict[str, Optional[str]], goal: str) -> List[str]:
    path = []
    current = goal
    while current is not None:
        path.append(current)
        current = parent[current]
    path.reverse()
    return path

## Task 2.1 — Implement BFS

Complete the function below.

### Required design

- Use `collections.deque`.
- Use FIFO behavior.
- Maintain a set of states that have already been discovered.
- Record parent links so that the final path can be reconstructed.
- Stop when the goal is removed from the frontier for expansion.

In [ ]:
def bfs(graph: Dict[str, List[str]], start: str, goal: str):
    frontier = deque([start])
    discovered = {start}
    parent = {start: None}
    expansion_order = []

    while frontier:
        # TODO 1: remove the next BFS state from the frontier
        current = None

        # TODO 2: record current in expansion_order

        # TODO 3: if current is the goal, return
        # (reconstruct_path(parent, goal), expansion_order)

        # TODO 4: inspect each neighbor in graph[current]
        # If it has not been discovered:
        #   - mark it discovered
        #   - store its parent
        #   - add it to the frontier
        pass

    return None, expansion_order

### BFS self-check

Before running the tests, state your prediction:

- **Expected returned path:**  
- **Expected first four expanded states:**  

Then run the tests.

In [ ]:
bfs_path, bfs_order = bfs(GRAPH, START, GOAL)

print("BFS path:", bfs_path)
print("BFS expansion order:", bfs_order)

assert bfs_path is not None
assert bfs_path[0] == "S"
assert bfs_path[-1] == "G"
assert bfs_path == ["S", "C", "G"]
assert bfs_order[:4] == ["S", "A", "B", "C"]

print("BFS tests passed.")

## Task 2.2 — Explain BFS in AI Terms

Answer in your own words.

1. What does `frontier` represent conceptually?
2. Why is `deque.popleft()` appropriate for BFS?
3. What does the `discovered` set prevent?
4. What information is stored in `parent`, and why is it useful?
5. Why is adding a state to `discovered` when it enters the frontier usually safer than waiting until much later?

**Your answers:**

## Task 2.3 — Manual BFS Trace vs. Program Trace

Compare your Task 1.1 prediction with the actual `bfs_order`.

- Which parts matched?
- If something differed, identify the **first step** where your prediction became incorrect.
- Was the cause conceptual, or was it due to the specified adjacency order?

**Your answer:**

# Part III — Depth-First Search

DFS uses a stack:

$$
\text{LIFO} = \text{Last In, First Out}.
$$

To preserve the graph's left-to-right adjacency order, neighbors should be pushed in the appropriate reverse order.

## Task 3.1 — Implement DFS

Complete the function below.

It must also return:

```python
(path, expansion_order)
```

In [ ]:
def dfs(graph: Dict[str, List[str]], start: str, goal: str):
    frontier = [start]
    discovered = {start}
    parent = {start: None}
    expansion_order = []

    while frontier:
        # TODO 1: pop the next DFS state
        current = None

        # TODO 2: record it

        # TODO 3: return the reconstructed path if it is the goal

        # TODO 4:
        # push neighbors so that DFS explores the adjacency list left-to-right.
        pass

    return None, expansion_order

### DFS self-check

Before execution, write:

- **Expected returned path:**  
- **Expected first four expanded states:**  

Then run the next cell.

In [ ]:
dfs_path, dfs_order = dfs(GRAPH, START, GOAL)

print("DFS path:", dfs_path)
print("DFS expansion order:", dfs_order)

assert dfs_path is not None
assert dfs_path[0] == "S"
assert dfs_path[-1] == "G"
assert dfs_order[:4] == ["S", "A", "D", "E"]

print("DFS tests passed.")

## Task 3.2 — Why Reverse the Neighbor Order?

Suppose you wrote:

```python
for child in graph[current]:
    frontier.append(child)
```

Given `graph["S"] == ["A", "B", "C"]`, which node would a stack pop next?

1. Predict the next state.
2. Explain why this may reverse the intended left-to-right traversal.
3. State one correct way to preserve the intended adjacency order.

**Your answer:**

## Task 3.3 — BFS vs. DFS on the Same Graph

Complete the table using your actual program output.

| Property | BFS | DFS |
|---|---|---|
| Returned path |  |  |
| Path length in actions |  |  |
| Expansion order |  |  |
| First branch emphasized |  |  |
| Frontier behavior | FIFO | LIFO |

Then explain:

> Why does a different frontier discipline change the returned solution even though both algorithms use the same graph and goal test?

**Your answer:**

# Part IV — Grid Search Experiment

We now compare BFS and DFS on a small grid.

Symbols:

- `S` = start
- `G` = goal
- `#` = obstacle
- `.` = free cell

In [ ]:
GRID = [
    "........",
    ".####...",
    "....#...",
    ".##.#.#.",
    "....#...",
    "........",
]

GRID_START = (5, 0)
GRID_GOAL = (0, 7)

MOVES = [
    ("Up", (-1, 0)),
    ("Right", (0, 1)),
    ("Down", (1, 0)),
    ("Left", (0, -1)),
]


def valid_grid_neighbors(state):
    r, c = state
    result = []

    for action, (dr, dc) in MOVES:
        nr, nc = r + dr, c + dc
        if (
            0 <= nr < len(GRID)
            and 0 <= nc < len(GRID[0])
            and GRID[nr][nc] != "#"
        ):
            result.append(((nr, nc), action))

    return result


def show_grid(path=None):
    canvas = [list(row) for row in GRID]

    if path:
        for r, c in path[1:-1]:
            if canvas[r][c] == ".":
                canvas[r][c] = "*"

    sr, sc = GRID_START
    gr, gc = GRID_GOAL
    canvas[sr][sc] = "S"
    canvas[gr][gc] = "G"

    print("\n".join(" ".join(row) for row in canvas))

show_grid()

## Task 4.1 — Generalize BFS and DFS to the Grid

Complete the two functions.

The graph is now generated dynamically by `valid_grid_neighbors(state)` rather than stored in a dictionary.

In [ ]:
def bfs_grid(start, goal):
    frontier = deque([start])
    discovered = {start}
    parent = {start: None}
    expansion_order = []

    while frontier:
        current = frontier.popleft()
        expansion_order.append(current)

        if current == goal:
            return reconstruct_grid_path(parent, goal), expansion_order

        for child, action in valid_grid_neighbors(current):
            if child not in discovered:
                discovered.add(child)
                parent[child] = current
                frontier.append(child)

    return None, expansion_order


def dfs_grid(start, goal):
    frontier = [start]
    discovered = {start}
    parent = {start: None}
    expansion_order = []

    while frontier:
        current = frontier.pop()
        expansion_order.append(current)

        if current == goal:
            return reconstruct_grid_path(parent, goal), expansion_order

        # TODO:
        # add children to the stack while preserving the MOVES ordering
        pass

    return None, expansion_order


def reconstruct_grid_path(parent, goal):
    path = []
    current = goal
    while current is not None:
        path.append(current)
        current = parent[current]
    return list(reversed(path))

## Task 4.2 — Predict Before Running the Experiment

Before executing BFS or DFS on the grid, answer:

1. Which algorithm do you expect to return the **shorter path in number of actions**?
2. Which one do you expect to expand more states on this particular grid?
3. Which prediction are you more confident about, and why?

Do **not** write “because BFS is better.” Refer to the search strategy.

**Your prediction:**

In [ ]:
bfs_grid_path, bfs_grid_order = bfs_grid(GRID_START, GRID_GOAL)
dfs_grid_path, dfs_grid_order = dfs_grid(GRID_START, GRID_GOAL)

print("BFS path length:", None if bfs_grid_path is None else len(bfs_grid_path) - 1)
print("BFS states expanded:", len(bfs_grid_order))
print("DFS path length:", None if dfs_grid_path is None else len(dfs_grid_path) - 1)
print("DFS states expanded:", len(dfs_grid_order))

print("\nBFS path:")
show_grid(bfs_grid_path)

print("\nDFS path:")
show_grid(dfs_grid_path)

## Task 4.3 — Experimental Comparison

Complete the table.

| Metric | BFS | DFS |
|---|---:|---:|
| Path length (actions) |  |  |
| States expanded |  |  |
| Was a solution found? |  |  |

Then answer:

1. Was your path-length prediction correct?
2. Did the algorithm with fewer expanded states also return the better solution?
3. Does this one experiment prove that one algorithm is always faster? Why not?
4. Which metric in this table measures **solution quality** for this unweighted grid?
5. Which metric is evidence about **search effort**?

**Your answers:**

# Part V — Debugging Common Search Errors

## Task 5.1 — Faulty BFS

A student writes:

```python
def faulty_bfs(graph, start, goal):
    frontier = [start]

    while frontier:
        current = frontier.pop()

        if current == goal:
            return current

        for child in graph[current]:
            frontier.append(child)
```

Identify **at least three problems**.

Possible categories to inspect:

- frontier discipline;
- repeated states;
- returned information;
- path reconstruction;
- behavior on cyclic graphs.

For each problem:

1. identify the faulty line or missing mechanism;
2. explain the AI consequence;
3. state the correction.

**Your answer:**

## Task 5.2 — Why Duplicate Detection Matters

Consider this cyclic graph:

```text
A → B → C
↑       ↓
└───────┘
```

If an uninformed graph-search algorithm never records visited/discovered states:

1. what can happen?
2. is the problem caused by BFS/DFS itself, or by the missing graph-search mechanism?
3. why is this distinction important?

**Your answer:**

# Part VI — Personalized Mini-Experiment

To reduce identical submissions, use the **last digit of your student ID** to select one goal.

- Last digit `0–3` → goal `(0, 5)`
- Last digit `4–6` → goal `(0, 6)`
- Last digit `7–9` → goal `(0, 7)`

Set the value below manually.

In [ ]:
LAST_DIGIT = None  # TODO: replace with an integer from 0 to 9

if LAST_DIGIT is not None:
    if 0 <= LAST_DIGIT <= 3:
        PERSONAL_GOAL = (0, 5)
    elif 4 <= LAST_DIGIT <= 6:
        PERSONAL_GOAL = (0, 6)
    elif 7 <= LAST_DIGIT <= 9:
        PERSONAL_GOAL = (0, 7)
    else:
        raise ValueError("LAST_DIGIT must be between 0 and 9")

    print("Your assigned goal is:", PERSONAL_GOAL)

## Task 6.1 — Predict, Run, Explain

Before running your personalized experiment:

- **My assigned goal:**  
- **I predict BFS path length will be:**  
- **I predict DFS will expand fewer / more / the same number of states as BFS:**  
- **Reason:**  

Then run both algorithms using your assigned goal.

In [ ]:
if LAST_DIGIT is not None:
    p_bfs_path, p_bfs_order = bfs_grid(GRID_START, PERSONAL_GOAL)
    p_dfs_path, p_dfs_order = dfs_grid(GRID_START, PERSONAL_GOAL)

    print("Personalized BFS path length:", len(p_bfs_path) - 1)
    print("Personalized BFS states expanded:", len(p_bfs_order))
    print("Personalized DFS path length:", len(p_dfs_path) - 1)
    print("Personalized DFS states expanded:", len(p_dfs_order))

### Task 6.2 — Explain the Personalized Result

1. Was your prediction correct?
2. Why did changing the goal affect the search behavior?
3. Did the state space itself change, or only the goal condition?
4. If two students have different assigned goals, should their traces necessarily be identical? Explain.

**Your answer:**

# Part VII — Individual Understanding Check

Your instructor may ask one short question about your notebook.

Possible prompts:

- Show me the line that makes BFS FIFO.
- Show me the line that makes DFS LIFO.
- Why do you need a `discovered` set?
- What is the difference between a frontier and an explored/discovered set?
- Why is the returned BFS path shallowest in an unweighted graph?
- Why did you reverse the neighbor order in DFS?
- If I change the goal, which part of the problem formulation changes?
- What would happen on a cyclic graph without duplicate detection?

> You are expected to explain the **AI concept represented by the code**, not memorize Python syntax.

# Reflection

Answer concisely but precisely.

### R1 — Completeness
Why is BFS complete when the branching factor is finite and a solution exists at finite depth?

**Answer:**

### R2 — Optimality
Why is BFS optimal for **unit step costs**, but not necessarily for unequal costs?

**Answer:**

### R3 — DFS Risk
Why can DFS spend a long time exploring an unproductive branch?

**Answer:**

### R4 — Memory
Why can BFS require much more memory than DFS?

**Answer:**

### R5 — Strategy Selection
Give one problem characteristic that would make BFS attractive, and one that would make DFS attractive.

**Answer:**

# Submission Checklist

Before submitting, verify that your notebook contains:

- [ ] BFS and DFS predictions before execution;
- [ ] completed manual traces;
- [ ] working BFS implementation;
- [ ] working DFS implementation;
- [ ] explanations of frontier, discovered set, and parent links;
- [ ] BFS vs. DFS comparison on the graph;
- [ ] completed grid experiment;
- [ ] interpretation of path length and expanded-state results;
- [ ] faulty BFS diagnosis;
- [ ] duplicate-detection explanation;
- [ ] personalized experiment using your assigned goal;
- [ ] prediction made before the personalized run;
- [ ] reflection answers;
- [ ] visible outputs from important code cells.

Suggested filename:

```text
Lab02_StudentID.ipynb
```

# Assessment Guide — 10 Marks

| Component | Marks | Evidence expected |
|---|---:|---|
| **Correct implementation** | **2.0** | BFS and DFS operate correctly |
| **Algorithmic justification** | **3.0** | Correct explanation of FIFO/LIFO, duplicate handling, path reconstruction, strategy behavior |
| **Experimental analysis** | **2.0** | Interprets graph/grid results and compares solution quality with effort |
| **Trace / prediction / debugging** | **1.0** | Manual reasoning, predictions, and faulty-code diagnosis |
| **Individual understanding check** | **1.0** | Short explanation of a selected part of the student's own work |
| **Code quality & completeness** | **1.0** | Readable code, complete responses, required outputs |
| **Total** | **10.0** |  |

> **Key rule:** Correct code without adequate explanation earns only a limited portion of the marks.

## Key Takeaways

- **BFS** uses a FIFO frontier and explores by increasing depth.
- **DFS** uses a LIFO frontier and follows one branch deeply before backtracking.
- The frontier policy changes the **expansion order** and may change the returned solution.
- Duplicate detection is essential for graph search, especially with cycles.
- BFS gives a shortest-action solution when all step costs are equal.
- DFS may use less memory but can spend substantial effort in an unproductive branch.
- Search experiments should distinguish **solution quality** from **search effort**.

The next lab will extend uninformed search to **Uniform-Cost Search and search-performance analysis**.